# SABRE BioSTEAM sandbox
Purpose:
- testing code that was made
- not necessarily the final simulation
- will need to make a new notebook for the final simulation

In [18]:
import importlib
import biosteam as bst

import sabre.chemicals as chem
import sabre.units.ad as ad_unit
import sabre.units.biogas_upgrading as up_unit
import sabre.units.centrifuge as centrifuge_unit
import sabre.systems.ad_biogas_system as sysmod

# --- reload modules ---
importlib.reload(chem)
importlib.reload(ad_unit)
importlib.reload(up_unit)
importlib.reload(centrifuge_unit)
importlib.reload(sysmod)

# --- reset flowsheet ---
bst.main_flowsheet.clear()

# --- set thermo ---
chem.create_chemicals()

# --- build + simulate ---
sys = sysmod.create_ad_biogas_system(quality="pelagic_high_quality")
sys.simulate(design_and_cost=True)
sys.show()

print("System:", sys.ID)
print("Units:", [u.ID for u in sys.units])
print("Streams:", [s.ID for s in sys.streams])

F = bst.main_flowsheet.stream
U = bst.main_flowsheet.unit

def get_stream(name):
    return getattr(F, name, None)

def safe_imass(stream, IDs):
    if stream is None: return None
    return sum(float(stream.imass[i]) for i in IDs if i in stream.chemicals.IDs)

def safe_imol(stream, ID):
    if stream is None: return None
    return float(stream.imol[ID]) if ID in stream.chemicals.IDs else 0.0

def pct(num, den):
    if den is None or den == 0: return None
    return num / den

TS_ids = ["Cellulose", "Ash"]

# --- optional pre-processing streams (only if they exist in your flowsheet) ---
feed    = get_stream("sargassum_feed")
cake    = get_stream("pressed_cake")
pressate= get_stream("pressate")
milled  = get_stream("milled_biomass")
losses  = get_stream("milling_losses")

# --- separator outlets (these SHOULD exist if you wired them) ---
soil = get_stream("soil_amendment")
liq  = get_stream("liquid_digestate")

# --- upgrading outlets (these SHOULD exist if you wired them) ---
bm  = get_stream("biomethane")
off = get_stream("offgas")
biogas = get_stream("biogas")

# -----------------------
# MASS CLOSURE CHECKS
# -----------------------
print("\n=== MASS CLOSURE CHECKS ===")
if feed is not None:
    print("Feed mass (kg/hr):", float(feed.F_mass))
else:
    print("Feed mass (kg/hr): <stream 'sargassum_feed' not found>")

if feed is not None and cake is not None and pressate is not None and feed.F_mass:
    print("Press closure:", (cake.F_mass + pressate.F_mass) / feed.F_mass)

if cake is not None and milled is not None and losses is not None and cake.F_mass:
    print("Mill closure:", (milled.F_mass + losses.F_mass) / cake.F_mass)

# -----------------------
# PRESS CHECK (only if you still have a press upstream)
# -----------------------
print("\n=== PRESS CHECK (if present) ===")
if cake is None or pressate is None:
    print("Pressed-cake/pressate streams not found; skipping.")
else:
    TS_cake = safe_imass(cake, TS_ids)
    TS_feed = safe_imass(feed, TS_ids) if feed is not None else None
    print("Cake mass (kg/hr):", float(cake.F_mass))
    print("Pressate mass (kg/hr):", float(pressate.F_mass))
    print("Cake TS (kg/hr):", TS_cake)
    print("Feed TS (kg/hr):", TS_feed)
    print("TS capture to cake:", pct(TS_cake, TS_feed))
    print("Cake TS wt%:", pct(TS_cake, cake.F_mass))
    if "Water" in cake.chemicals.IDs and cake.F_mass:
        print("Cake moisture wt%:", float(cake.imass["Water"]) / cake.F_mass)

# -----------------------
# AD DESIGN/COST
# -----------------------
AD = getattr(U, "AD", None)
print("\n=== AD DESIGN ===")
if AD is None:
    print("<unit 'AD' not found>")
else:
    for k, v in AD.design_results.items():
        print(f"{k}: {v}")

print("\n=== AD COST ===")
if AD is None:
    print("<unit 'AD' not found>")
else:
    print("Baseline purchase costs:", dict(AD.baseline_purchase_costs))
    print("Installed cost:", getattr(AD, "installed_cost", None))

# -----------------------
# UPGRADING CHECK
# -----------------------
UP = getattr(U, "UP", None)
print("\n=== UPGRADING CHECK ===")
if UP is None or bm is None or off is None:
    print("UP/bio streams not found; skipping.")
else:
    bm_ch4 = safe_imol(bm, "Methane") or 0.0
    bm_co2 = (safe_imol(bm, "CarbonDioxide") or 0.0) + (safe_imol(bm, "CO2") or 0.0)
    off_ch4 = safe_imol(off, "Methane") or 0.0
    off_co2 = (safe_imol(off, "CarbonDioxide") or 0.0) + (safe_imol(off, "CO2") or 0.0)

    print("Biomethane (kmol/hr): CH4 =", bm_ch4, ", CO2 =", bm_co2)
    print("Offgas (kmol/hr):     CH4 =", off_ch4, ", CO2 =", off_co2)

    if biogas is not None:
        ch4_in = safe_imol(biogas, "Methane") or 0.0
        if ch4_in > 0:
            print("CH4 recovery (approx):", bm_ch4 / ch4_in)

    print("UP design_results:", getattr(UP, "design_results", {}))
    print("UP baseline_purchase_costs:", getattr(UP, "baseline_purchase_costs", {}))

# -----------------------
# DECANTER CENTRIFUGE CHECK
# -----------------------
C = getattr(U, "C", None) or getattr(U, "DigestateCentrifuge", None)
print("\n=== DECANTER CENTRIFUGE CHECK ===")
if C is None:
    print("<centrifuge unit not found; expected U.C or U.DigestateCentrifuge>")
else:
    # inlet stream to centrifuge
    cin = C.ins[0] if len(C.ins) else None
    print("Digestate in (kg/hr):", float(cin.F_mass) if cin is not None else None)

    if soil is None or liq is None:
        print("<soil_amendment or liquid_digestate stream not found>")
    else:
        print("Soil amend out (kg/hr):", float(soil.F_mass))
        print("Liquid digestate out (kg/hr):", float(liq.F_mass))
        if cin is not None and cin.F_mass:
            print("Centrifuge mass closure:", (soil.F_mass + liq.F_mass) / cin.F_mass)

        TS_soil = safe_imass(soil, TS_ids)
        TS_liq  = safe_imass(liq, TS_ids)
        print("Soil TS (kg/hr):", TS_soil, " | Liquid TS (kg/hr):", TS_liq)

        if "Water" in soil.chemicals.IDs and soil.F_mass:
            print("Soil moisture wt%:", float(soil.imass["Water"]) / soil.F_mass)
        if "Water" in liq.chemicals.IDs and liq.F_mass:
            print("Liq moisture wt%:", float(liq.imass["Water"]) / liq.F_mass)

    # design + cost outputs from the unit itself
    print("\nCentrifuge design_results:", getattr(C, "design_results", {}))
    print("Centrifuge baseline_purchase_costs:", getattr(C, "baseline_purchase_costs", {}))


System: AD_Biogas_sys
ins...
[0] sargassum_feed  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      3.01e+04
                    Cellulose  335
                    Ash        2.86e+04
outs...
[0] milling_losses  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      1.26e+03
                    Cellulose  49.2
                    Ash        4.21e+03
[1] biomethane  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Methane        77.5
                    CarbonDioxide  2.61
[2] offgas  
    phase: 'g', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Methane        0.783
                    CarbonDioxide  49.6
[3] pressate  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      2.17e+04
                    Cellulose  6.7
                    Ash        573
[4] soil_amendment  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water      6.03e+03
                    Cellulose  109
                    Ash   